# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at:
- https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata
metadata = dataset.metadata.to_json()

print("Dataset Name: {}".format(metadata.get('name', 'N/A')))
print("Description: {}".format(metadata.get('description', 'N/A'))) 

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will enumerate the record sets, fields, and columns using their `@id` values.

In [ ]:
# List available record sets and fields (referenced by @id)
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for field in fields:
            print(f"    - Field @id: {field['@id']} | Name: {field.get('name', 'N/A')}")
            columns = field.get('column', [])
            if columns:
                print("      Columns:")
                for col in columns:
                    print(f"        - Column @id: {col['@id']} | Name: {col.get('name', 'N/A')}")
    print("")
if not record_sets:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Record sets, fields, and columns are referenced using their `@id` values as found above.

In [ ]:
# Extract data from each record set
# First, get the list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        # Load records (each record is a dictionary mapping field @id to value)
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet @id: {record_set_id}")
        print(f"Columns (field @ids): {df.columns.tolist()}")
    except Exception as e:
        print(f"Error loading records from RecordSet @id: {record_set_id}: {e}")

# For demonstration, display the head of the first available dataframe
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"Preview of RecordSet {first_rs_id}:")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We will select numeric fields for analysis and demonstrate filtering, normalization, and grouping operations.

Use field and column `@id`s from the data overview.

In [ ]:
# Identify a numeric field and a grouping field from the first RecordSet
first_record_set = record_sets[0] if record_sets else None

if first_record_set and first_record_set.get('field'):
    field_ids = [field['@id'] for field in first_record_set['field']]
    numeric_field_id = None
    group_field_id = None
    # Heuristically pick first Integer or Float and one categorical field
    for field in first_record_set['field']:
        if field.get('dataType') in ['schema:Float', 'schema:Integer']:
            numeric_field_id = field['@id']
            break
    for field in first_record_set['field']:
        # Pick a Text, DefinedTerm, or similar
        if field.get('dataType') in ['schema:Text', 'schema:DefinedTerm']:
            group_field_id = field['@id']
            break

    rs_id = first_record_set['@id']
    df = dataframes.get(rs_id, None)
    if df is not None and numeric_field_id in df.columns:
        # Filter records where numeric field > threshold
        threshold = df[numeric_field_id].mean() if numeric_field_id else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records ({numeric_field_id} > {threshold}): {len(filtered_df)} rows")

        # Normalize the numeric field
        filtered_df[numeric_field_id + '_normalized'] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()

        print("Normalized numeric values (preview):")
        print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

        # Group and aggregate by group_field if possible
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No suitable numeric field available for EDA.")
else:
    print("No fields available for EDA.")

## 5. Visualization
Visualize distributions and relationships between fields (using `matplotlib`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization for demonstration
if first_record_set and df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    # If grouping field available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization not possible, missing numeric or categorical fields.")

## 6. Conclusion
This notebook demonstrated loading a FAIR^2-compliant dataset via `mlcroissant`, referencing record sets and fields by `@id`, and performing basic EDA and visualization.

- Data was loaded using the Croissant schema and examined for structure.
- References to record sets, fields, and columns were consistently managed by their `@id`.
- Filtering, normalization, and grouping operations illustrated key exploratory steps.
- Visualizations supported further insights into the dataset's numeric and categorical distributions.
- The FAIR^2 dataset provides rich, actionable survey data for rangeland management adoption studies in Northern Kenya.